# Value Investing AI Agent Framework: System Architecture & Design

## 1. System Architecture & Core Modules

### 1.1 Data Ingestion & Engineering Pipeline
- **Fundamental Data Sourcing & Cleaning:**
  - **Sources:** Access API endpoints (e.g., Financial Modeling Prep, Alpha Vantage, sec-api, or yfinance) for historical income statements, balance sheets, and cash flow statements.
  - **Cleaning:** Standardize accounting metrics (e.g., harmonizing EBIT, EBITDA, Free Cash Flow definitions across companies). Handle missing data by forward-filling or omitting incomplete fiscal years. Convert all absolute numbers to per-share metrics where applicable.
- **Unstructured Data Pipelines (RAG Approach):**
  - **Sources:** 10-K/10-Q filings, earnings call transcripts, management discussion and analysis (MD&A).
  - **Processing:**
    - Extract text using tools like `sec-api` and PDF parsers.
    - Chunk text into semantic sections (e.g., Risk Factors, Management Discussion).
    - Embed chunks using a financial domain-specific embedding model (e.g., FinBERT embeddings or OpenAI's `text-embedding-3-large`).
    - Store in a Vector Database (e.g., ChromaDB, Pinecone, or Qdrant).
  - **Retrieval:** When the agent evaluates a company, it queries the RAG system for specific qualitative aspects like "competitor threats," "management outlook," and "regulatory risks."

### 1.2 The Value Investing AI Agent
- **Intrinsic Value Calculation:**
  - The agent utilizes a deterministic calculation engine for quantitative valuation to avoid LLM hallucination.
  - **Discounted Cash Flow (DCF):** Projects future free cash flows based on historical growth rates (adjusted by LLM qualitative sentiment) and discounts them back to present value using the Weighted Average Cost of Capital (WACC).
  - **Graham Number:** Calculates the maximum fair value based on Earnings Per Share (EPS) and Book Value Per Share (BVPS).
- **LLM Qualitative Assessment (Moat & Management):**
  - The LLM acts as the "Qualitative Analyst." It is prompted with summarized information retrieved from the RAG pipeline.
  - **Prompting Strategy:** Analyze for "Economic Moats" (network effects, switching costs, cost advantages), "Management Competence" (capital allocation history, CEO tenure), and "Competitive Advantage."
- **Decision-Making Guardrails (Margin of Safety):**
  - **Strict Rule:** `Buy Signal = True IF Current_Price < (Intrinsic_Value * (1 - Margin_of_Safety_Percentage))`
  - The default Margin of Safety is set to 30%. The LLM cannot override this rule; it can only influence the inputs to the Intrinsic Value calculation.

### 1.3 The Backtesting Engine
- **Event-Driven Architecture:** The engine steps through historical dates daily, triggering the AI Agent only on specific events: Earnings releases, 10-K filings, or significant price drops.
- **Addressing Quantitative Pitfalls:**
  - **Survivorship Bias:** Must use a historical stock universe dataset that includes delisted or bankrupt companies.
  - **Look-Ahead Bias:** Point-in-time data is critical. The system must only provide the agent with data that was publicly available *on that specific historical date*.
  - **Transaction Fees & Slippage:** Model realistic market impact and broker commissions (e.g., 0.1% slippage per trade + $1 commission).
- **Execution Logic & Portfolio Sizing:**
  - **Rebalancing:** Quarterly or Annual rebalancing.
  - **Sizing:** Conviction-based weighting capped at a maximum position size (e.g., 5%–10% per stock).

### 1.4 Performance Analytics & Risk Management
- **Core Metrics:** CAGR, Sharpe Ratio, Sortino Ratio, Maximum Drawdown, Alpha & Beta, Win/Loss Ratio.
- **Risk Management Protocols:**
  - **Sector Limits:** Maximum 20–25% exposure to any single sector.
  - **Max Position Size:** Strict cap of 5–10% of portfolio NAV per equity.
  - **Stop-Loss:** Sell if fundamental thesis is broken (massive downgrade in intrinsic value).

## 2. Recommended Technology Stack

| Category | Tools |
|---|---|
| Language | Python 3.10+ |
| Agent Framework | LangChain / LangGraph, AutoGen (multi-agent) |
| LLMs | OpenAI `gpt-4o`, Anthropic `claude-3-5-sonnet` |
| Market Data | `yfinance`, Alpaca, Polygon.io |
| Fundamentals | `sec-api`, Financial Modeling Prep, AlphaVantage |
| Vector DB | `ChromaDB` (local), `Pinecone` (production) |
| Backtesting | Backtrader, Zipline-Reloaded |
| Analytics | `pandas`, `numpy`, `pyfolio` |

## 3. High-Level Folder Structure

```
value_investing_framework/
│
├── data/
│   ├── fundamental/
│   ├── market/
│   └── unstructured/
│
├── ingestion/
│   ├── market_data.py
│   ├── sec_filings.py
│   └── vector_store.py
│
├── agent/
│   ├── prompts/
│   ├── valuation_models.py
│   ├── llm_analyst.py
│   └── value_agent.py
│
├── backtest/
│   ├── engine.py
│   ├── point_in_time.py
│   ├── execution.py
│   └── analytics.py
│
├── config/
│   └── settings.yaml
│
├── main.py
└── requirements.txt
```

## 4. Core Agent Implementation

In [ ]:
import pandas as pd
from typing import Dict, Any


class ValueInvestingAgent:
    def __init__(self, llm_client, vector_db, margin_of_safety: float = 0.30):
        """
        Initializes the Value Agent with an LLM client, access to the RAG database,
        and a strict Margin of Safety threshold.

        Args:
            llm_client: An initialized LLM client (e.g., OpenAI or LangChain wrapper).
            vector_db: A vector database client for RAG retrieval (e.g., ChromaDB).
            margin_of_safety: The required discount to intrinsic value before buying (default: 30%).
        """
        self.llm = llm_client
        self.rag_db = vector_db
        self.margin_of_safety = margin_of_safety

    def calculate_intrinsic_value(self, financials: pd.DataFrame) -> float:
        """
        Deterministic calculation of Intrinsic Value using the Graham Number.
        Formula: sqrt(22.5 * EPS * BVPS)

        Args:
            financials: DataFrame containing at minimum 'eps' and 'book_value_per_share' columns.

        Returns:
            The estimated intrinsic value as a float, or 0.0 if calculation is not possible.
        """
        eps = financials['eps'].iloc[-1]
        bvps = financials['book_value_per_share'].iloc[-1]

        if eps > 0 and bvps > 0:
            graham_number = (22.5 * eps * bvps) ** 0.5
            return graham_number
        return 0.0

    def assess_qualitative_moat(self, ticker: str, date: str) -> Dict[str, Any]:
        """
        Uses RAG to fetch historical text data and queries the LLM for qualitative metrics.
        Enforces point-in-time constraint to prevent look-ahead bias.

        Args:
            ticker: The stock ticker symbol (e.g., 'AAPL').
            date: The evaluation date string (e.g., '2023-01-15') for point-in-time filtering.

        Returns:
            A dict with 'moat_score' (1-10), 'management_score' (1-10), and 'thesis_summary'.
        """
        # Retrieve relevant historical context from Vector DB (point-in-time safe)
        context = self.rag_db.query(
            filter={"ticker": ticker, "date_lte": date},
            query_texts=["management competence", "competitive moat", "risk factors"]
        )

        # Prompt LLM to analyze the retrieved context
        prompt = f"""
        You are a Value Investing Analyst. Based on the following SEC filings and transcripts:
        {context}

        Assess the company's economic moat and management quality.
        Return a JSON with:
        - 'moat_score' (1-10): Strength of durable competitive advantages.
        - 'management_score' (1-10): Quality and honesty of management.
        - 'thesis_summary': A concise 2-3 sentence investment thesis summary.
        """

        response = self.llm.invoke(prompt)
        return response.json()

    def generate_signal(self, ticker: str, current_price: float,
                        financials: pd.DataFrame, date: str) -> Dict[str, Any]:
        """
        Main decision-making function. Combines quantitative valuation with
        qualitative LLM assessment and enforces the Margin of Safety guardrail.

        Args:
            ticker: The stock ticker symbol.
            current_price: The current market price of the stock.
            financials: DataFrame of fundamental financial data.
            date: The evaluation date (used for point-in-time RAG filtering).

        Returns:
            A dict with 'action' (BUY or HOLD), and supporting analysis details.
        """
        # Step 1: Quantitative Valuation (deterministic, no LLM)
        intrinsic_value = self.calculate_intrinsic_value(financials)

        if intrinsic_value == 0:
            return {"action": "HOLD", "reason": "Negative earnings or book value prevents valuation."}

        # Step 2: Guardrail — Margin of Safety Check
        target_buy_price = intrinsic_value * (1 - self.margin_of_safety)

        if current_price > target_buy_price:
            return {
                "action": "HOLD",
                "reason": f"Price ${current_price:.2f} is above target buy price ${target_buy_price:.2f}."
            }

        # Step 3: Qualitative Verification (only runs if price is attractive)
        qual_analysis = self.assess_qualitative_moat(ticker, date)

        if qual_analysis['moat_score'] < 6 or qual_analysis['management_score'] < 6:
            return {
                "action": "HOLD",
                "reason": "Value trap detected. Weak qualitative fundamentals despite low price."
            }

        # Step 4: Final Buy Signal — all conditions met
        return {
            "action": "BUY",
            "confidence": (target_buy_price - current_price) / target_buy_price,
            "intrinsic_value": intrinsic_value,
            "target_buy_price": target_buy_price,
            "analysis": qual_analysis['thesis_summary']
        }

## 5. Usage Example (Dry Run)

In [ ]:
# Simulate financial data for demonstration purposes
sample_financials = pd.DataFrame({
    'eps': [2.5, 3.1, 3.8, 4.2],                    # Earnings per share (last 4 years)
    'book_value_per_share': [18.0, 20.5, 22.1, 24.3] # Book value per share
})

# Instantiate the agent with placeholder clients
# Replace `None` with your actual llm_client and vector_db in production
agent = ValueInvestingAgent(llm_client=None, vector_db=None, margin_of_safety=0.30)

# Calculate intrinsic value from financial data alone (no LLM needed)
intrinsic_value = agent.calculate_intrinsic_value(sample_financials)
target_buy_price = intrinsic_value * (1 - agent.margin_of_safety)

print(f"Intrinsic Value (Graham Number): ${intrinsic_value:.2f}")
print(f"Target Buy Price (30% MoS):      ${target_buy_price:.2f}")
print()
print("To generate a full BUY/HOLD signal with qualitative analysis,")
print("call: agent.generate_signal(ticker, current_price, financials, date)")